# Clase 8 — Un mini-GPT propio, y la traducción a ChatGPT

**Deep Learning para Cientistas Sociales**

> **La idea de hoy en una línea.** Un modelo de lenguaje es una máquina que dibuja
> **la probabilidad de la próxima ficha**, condicionada por lo que hay en la ventana.
> No hay una base de datos adentro, no hay un verificador de verdad, y no hay
> nada adentro que detecte cuándo no sabe. Todo lo que te vuelve loco de ChatGPT
> sale de ahí.

---

**Cuando termines esta clase vas a poder:**

1. Entrenar un modelo de lenguaje desde cero y leer su curva de loss.
2. Explicar qué significa «predecir la próxima ficha» y por qué de ahí sale todo lo demás.
3. Explicar, en términos del modelo que acabás de entrenar y no por analogía vaga:
   por qué alucina, por qué le importa tanto el prompt, qué es la ventana de contexto,
   qué hace la temperatura y por qué cuenta mal las letras.
4. Decidir con criterio cuándo un LLM es una herramienta válida para una tarea de
   investigación tuya, y qué tenés que reportar si la usás.

**Lo que hace falta traer:** la clase 7. El bloque transformer que armaste ahí es,
literalmente, la pieza que hoy vamos a apilar.

---

> ## ⏱ Antes que nada, leé esto
>
> La **sección 5** entrena el modelo y **tarda entre 4 y 7 minutos en CPU**.
> Los mismos 1200 pasos se cronometraron **seis veces** en la misma máquina de 4 hilos
> y dieron **243 s, 267 s, 280 s, 297 s, 360 s y 432 s**: la CPU se comparte con todo lo
> que tengas abierto, así que el número casi se duplica según el día. (En una
> computadora reciente puede tardar bastante menos: con un procesador ARM nuevo tardó
> unos 2 minutos.) **Contá con siete
> minutos.** Arrancala apenas llegues y **seguí leyendo mientras corre**: la sección 6
> está puesta ahí justamente para llenar ese rato. No te quedes mirando la barra.
>
> Si tenés GPU (en Colab: *Entorno de ejecución → Cambiar tipo de entorno → T4*),
> el notebook lo detecta solo y sube los pasos, porque le sobra tiempo. Ojo: con GPU
> (o con otra computadora, u otra versión de torch) tus muestras **no** van a ser
> letra por letra las de las slides. Es normal: la sección 5 explica por qué.

**Cómo se usa este notebook.** Corré las celdas en orden (`Shift + Enter`). Donde diga
`TODO` te toca a vos.

## 0. Preparar el entorno

Esta celda es **idéntica en los diez notebooks del curso**. En Colab clona el
repo y deja todo a mano; en tu máquina no hace prácticamente nada porque ya
tenés los archivos. Corrémela y seguí de largo: no hay nada que entender acá.

In [ ]:
# =============================================================================
# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.
# En Colab: clona el repo del curso (o usa el .zip que subas) e instala lo que falte.
# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.
# =============================================================================
import os
import pathlib
import subprocess
import sys
import zipfile

REPO = "https://github.com/johannperez/dlss"
RAMA = "main"
CARPETA = "dlss"


def en_colab():
    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def buscar_raiz(desde=None):
    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""
    punto = pathlib.Path(desde or os.getcwd()).resolve()
    for candidata in [punto, *punto.parents]:
        if (candidata / "_shared" / "ssdl").is_dir():
            return candidata
    return None


def raiz_desde_zip():
    """Plan B: si subiste el .zip del repo (dlss-main.zip o parecido), lo abre."""
    for archivo in sorted(pathlib.Path(".").glob("dlss*.zip")):
        with zipfile.ZipFile(archivo) as z:
            z.extractall(CARPETA + "-zip")
        for ssdl in pathlib.Path(CARPETA + "-zip").glob("**/_shared/ssdl"):
            return ssdl.parent.parent.resolve()
    return None


raiz = buscar_raiz()

if raiz is None and en_colab():
    if not pathlib.Path(CARPETA).exists():
        print("Bajando el repo del curso (una sola vez por sesión)...")
        clon = subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                               REPO, CARPETA],
                              env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
        if clon.returncode != 0:
            print("No pude clonar el repo. Pruebo con un .zip subido a mano...")
    raiz = buscar_raiz(CARPETA) if pathlib.Path(CARPETA).exists() else None
    raiz = raiz or raiz_desde_zip()

if raiz is None:
    raise SystemExit(
        "No encuentro el repo del curso.\n"
        "· En Colab, si el clone falló (por ejemplo, «Repository not found» o 404):\n"
        "  el repo github.com/johannperez/dlss tiene que ser público. Si no lo es,\n"
        "  o no hay red hacia GitHub, conseguí el .zip del repo (en la página del\n"
        "  repo: Code → Download ZIP; o pedíselo a quien da el curso), subilo con\n"
        "  el panel de archivos de Colab (el nombre tiene que empezar con `dlss`,\n"
        "  como dlss-main.zip) y volvé a correr esta celda.\n"
        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`.\n"
        "Más detalle en recursos/setup.md."
    )

# Dependencias. En Colab casi todas vienen puestas: instalamos sólo lo que falte.
# En tu máquina no instalamos nada (torch es pesado y varias clases no lo usan):
# si falta algo, avisamos y lo instalás vos (ver recursos/setup.md).
faltan = []
for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),
                        ("torch", "torch")]:
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)
if faltan and en_colab():
    print(f"Instalando {', '.join(faltan)}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan],
                   check=True)
elif faltan:
    print(f"[!] Falta {', '.join(faltan)}. Si este notebook lo usa, instalalo con:\n"
          f"    {sys.executable} -m pip install {' '.join(faltan)}")

sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`
os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos

print(f"Listo. Repo del curso en: {raiz}")

### Imports y semilla

Fijamos la semilla en 1983 **siempre**. Hoy importa más que nunca: generar texto es
un **sorteo**, y sin semilla fija vos y la persona de al lado no ven el mismo texto,
así que no pueden discutir sobre lo mismo. (Esto va a volver, con nombre y apellido,
en la sección 7.)

Una salvedad que también es contenido: la semilla fija da **exactamente** el mismo
resultado en la misma computadora y con las mismas versiones de las librerías. Con
otra máquina u otra versión de torch, los números cambian un poco, y el texto
generado puede cambiar bastante. Lo vas a ver en la sección 5.

In [ ]:
import json
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from ssdl import datos, graficos as g, utils

utils.fijar_semilla(1983)          # la semilla del curso
utils.resumen_del_entorno()


def revisar(condicion, si_va, si_no):
    """Verificación amable: dice qué mirar cuando no sale, sin stack trace.

    No levanta excepciones a propósito: un `assert` que explota en el medio de
    una clase asusta y no explica nada. Esto dice qué mirar.
    """
    print(("✓ " + si_va) if condicion else ("✗ " + si_no))

---

## 1. La idea: el objetivo de entrenamiento más tonto del mundo

Todo lo que hace un modelo de lenguaje es esto:

> Dado un pedazo de texto, **poné una probabilidad a cada ficha posible** de ser la
> que sigue.

Nada más. No hay un módulo de razonamiento, no hay una base de datos de hechos, no
hay un verificador. Hay una función con un millón y medio de perillas que, para cada
posición del texto, dibuja una distribución de probabilidad.

**Por qué esto es tan poderoso.** Porque el dataset **se arma solo**. Para entrenar un
clasificador de titulares necesitás que alguien etiquete titulares, uno por uno. Para
entrenar un modelo de lenguaje sólo necesitás texto: cada posición del texto **es** su
propia etiqueta, porque la respuesta correcta es la ficha que efectivamente venía
después. Por eso se puede entrenar con internet entero y por eso los LLM existen.

**La analogía de ciencias sociales.** Pensalo como aprender un **género discursivo**.
Cualquiera que haya trabajado con análisis del discurso sabe que un debate
parlamentario tiene fórmulas de tratamiento, estructura de turnos, cadencia. Vas a ver
al modelo aprender **la forma antes que cualquier contenido**, y eso mismo pasa en los
modelos grandes: sólo que ahí está mucho mejor disimulado.

**Lo que NO es.** El modelo no «consulta» nada. Cuando te da un dato correcto, ese
dato está **destilado en los pesos**, mezclado con todos los demás, sin dirección ni
índice. Por eso no se le puede corregir un dato puntual como se corrige una celda de
una planilla.

`[CALLBACK DIBUJO]` En la clase 1 dibujábamos una recta con **2** perillas sobre 36
puntos. Hoy dibujamos, en cada posición del texto, una distribución sobre todas las
fichas posibles, con **1.541.056** perillas. **Es la misma máquina.** Misma loss
(clase 3), mismo descenso (clase 3), mismo ciclo (clase 4), mismos embeddings
(clase 6), mismo bloque (clase 7). Hoy no hay ninguna pieza nueva.

---

## 2. Miralo: el corpus

Vamos a entrenar sobre **Aguasclaras**: los debates parlamentarios de un país que no
existe.

> ⚠️ **CORPUS SINTÉTICO.** Aguasclaras es un país ficticio, con diputados ficticios,
> partidos ficticios y leyes ficticias. Lo generamos nosotros con
> `datos/generar_aguasclaras.py`, con semilla fija.
>
> **Por qué usamos esto y no discursos reales**, teniendo 8 MB de discursos reales a
> mano: porque tenemos **el padrón de la verdad**. Cuando el modelo se invente un
> diputado, vamos a poder abrir la lista completa de los que existen y mostrar, en
> cinco segundos y delante de todos, que ese no está. Con un corpus real habría que
> ir a chequear a mano cada afirmación — que es, justamente, el problema del que trata
> la segunda mitad de la clase.
>
> **Y el precio que se paga:** el vocabulario de Aguasclaras es pobre (sale de
> plantillas), así que **el modelo va a aprender más rápido y mejor de lo que
> aprendería con texto real**. No te lleves de acá la impresión de que esto es fácil.

In [ ]:
texto = datos.debates_aguasclaras()
verdad = datos.verdad_aguasclaras()          # el padrón: quién existe de verdad

print(f"corpus: {len(texto):,} caracteres "
      f"({len(texto.split()):,} palabras aproximadamente)")
print(f"padrón: {len(verdad['diputados'])} diputados · "
      f"{len(verdad['partidos'])} partidos · {len(verdad['leyes'])} leyes")
print()
print("--- un fragmento -------------------------------------------------------")
print(texto[3200:3900])

---

## 3. Fichas: partir el texto en pedazos

Para que el modelo lea, hay que darle números. La clase 6 mostró las tres estrategias:
por caracteres, por palabras, por sub-palabras (BPE).

**Acá vamos a usar el tokenizador por caracteres**, y conviene ser transparente con
esa decisión:

| | por caracteres (lo que usamos hoy) | por sub-palabras (lo que usan GPT, Claude, Llama) |
|---|---|---|
| vocabulario | 71 fichas | 50.000 a 200.000 fichas |
| secuencia | larguísima (una ficha por letra) | corta |
| palabras desconocidas | no existen nunca | no existen nunca |
| para aprender | converge lindo con poco corpus | necesita mucho más texto |

Elegimos caracteres porque con 1 MB de corpus y unos minutos de CPU es lo que **funciona**
y lo que deja ver la progresión completa: de sopa de letras a cadencia parlamentaria.
Cuando lleguemos a «por qué cuenta mal las letras» vamos a volver a las sub-palabras,
porque **esa** parte del comportamiento de ChatGPT depende de que use sub-palabras y no
caracteres. Lo decimos ahí de nuevo.

In [ ]:
class TokenizadorCaracteres:
    """Un carácter, una ficha. El tokenizador más simple que existe."""

    def __init__(self, texto):
        self.fichas = sorted(set(texto))                     # el vocabulario
        self.a_id = {c: i for i, c in enumerate(self.fichas)}
        self.a_ficha = {i: c for i, c in enumerate(self.fichas)}

    def __len__(self):
        return len(self.fichas)

    def codificar(self, s):
        return [self.a_id[c] for c in s if c in self.a_id]

    def decodificar(self, ids):
        return "".join(self.a_ficha[int(i)] for i in ids)


tok = TokenizadorCaracteres(texto)

print(f"vocabulario: {len(tok)} fichas")
print("son estas:", "".join(tok.fichas).replace("\n", "⏎"))
print()

prueba = "Señor presidente, pido la palabra."
print(f"«{prueba}»")
print("->", tok.codificar(prueba))
print("->", f"«{tok.decodificar(tok.codificar(prueba))}»")

revisar(tok.decodificar(tok.codificar(texto)) == texto,
        "ida y vuelta perfecta: codificar y decodificar el corpus entero "
        "devuelve exactamente el mismo texto.",
        "algo se pierde en la ida y vuelta: fijate que el vocabulario se haya "
        "construido sobre ESTE texto y no sobre otro.")

### El dataset se arma solo

Acá está el truco que hace posible todo esto. Cortamos una ventana de texto y la
respuesta correcta es **el mismo texto corrido un lugar**.

Y lo importante: una ventana de 8 fichas no es **un** ejemplo de entrenamiento, son
**ocho**. Con «S», la respuesta es «e»; con «Se», la respuesta es «ñ»; con «Señ», la
respuesta es «o»... El modelo aprende las ocho predicciones en una sola pasada. Por eso
es tan barato entrenar con texto y tan caro con datos etiquetados a mano.

In [ ]:
ids = torch.tensor(tok.codificar(texto), dtype=torch.long)

# Partimos por POSICIÓN, no al azar: el último 10% del corpus son sesiones que el
# modelo no va a ver nunca. (Clase 4: si partís al azar dentro de un texto continuo,
# el pedazo de validación está rodeado de contexto que sí viste. Eso es fuga.)
corte = int(0.9 * len(ids))
ids_train, ids_val = ids[:corte], ids[corte:]
print(f"train: {len(ids_train):,} fichas · validación: {len(ids_val):,} fichas")
print()

# Una ventana chiquita, para poder mirarla entera. Arrancamos justo en la «S» de
# «Señor pre...» — es el mismo ejemplo que está proyectado en la slide, así que lo
# que te sale acá tiene que coincidir letra por letra con lo que hay en pantalla.
ventana = 8
inicio = texto.index("Señor presidente", 30_000)
x_demo = ids[inicio:inicio + ventana]
y_demo = ids[inicio + 1:inicio + ventana + 1]

print(f"UNA ventana de {ventana} fichas contiene {ventana} ejemplos de entrenamiento:")
print()
for t in range(ventana):
    contexto = tok.decodificar(x_demo[:t + 1])
    objetivo = tok.decodificar([y_demo[t]])
    print(f"  cuando ve «{contexto:>9}»  ->  la respuesta correcta es «{objetivo}»")

In [ ]:
def sacar_lote(datos_ids, tam_lote, contexto, gen):
    """Corta `tam_lote` ventanas al azar. x es el contexto, y es lo que sigue."""
    i = torch.randint(len(datos_ids) - contexto - 1, (tam_lote,), generator=gen)
    x = torch.stack([datos_ids[j:j + contexto] for j in i])
    y = torch.stack([datos_ids[j + 1:j + contexto + 1] for j in i])
    return x, y


gen = torch.Generator().manual_seed(1983)
xb, yb = sacar_lote(ids_train, 4, 16, gen)
print("x (lo que ve):", tuple(xb.shape), " y (lo que tiene que predecir):", tuple(yb.shape))
print()
for fila in range(2):
    print(f"  x: «{tok.decodificar(xb[fila])}»")
    print(f"  y: «{tok.decodificar(yb[fila])}»   <- el mismo texto, corrido UNO")
    print()

### El modelo tonto: el bigrama

Antes del transformer, la línea de base más tonta posible: **mirar sólo la ficha
anterior**. Contamos, en todo el corpus, cuántas veces después de cada carácter viene
cada otro carácter. Eso es todo. Cero perillas aprendidas: son conteos.

In [ ]:
# Una tabla de 71 x 71: "después de la ficha A, ¿cuántas veces vino la ficha B?"
tabla = torch.zeros(len(tok), len(tok))
for a, b in zip(ids_train[:-1].tolist(), ids_train[1:].tolist()):
    tabla[a, b] += 1
probs_bigrama = tabla / tabla.sum(1, keepdim=True).clamp(min=1)

# Generamos con el bigrama: sorteamos la próxima ficha mirando SOLO la anterior
g_bi = torch.Generator().manual_seed(1983)
actual = tok.a_id["S"]
salida = []
for _ in range(300):
    actual = int(torch.multinomial(probs_bigrama[actual], 1, generator=g_bi))
    salida.append(actual)

# Su loss: qué tan sorprendido queda con el texto de validación
loss_bigrama = float(
    -torch.log(probs_bigrama[ids_val[:-1], ids_val[1:]].clamp(min=1e-9)).mean())

print("LO QUE ESCRIBE EL BIGRAMA:")
print(tok.decodificar(salida))
print()
print(f"loss del bigrama en validación: {loss_bigrama:.3f}")
print("(la loss es «qué tan sorprendido queda con el texto que no vio». "
      "Más chico = menos sorprendido.)")

**Miralo bien: ya escribe algo.** Hay sílabas del castellano, hay espacios en lugares
plausibles, hay comas. Con una tabla de conteos y ninguna perilla.

**Y miralo mejor: no puede funcionar.** Cuando llega a la letra `a`, lo único que sabe
es «vengo de una a». No sabe si está en el medio de «palabra», si el que habla es del
bloque tal, ni de qué ley se estaba hablando hace diez renglones. **No tiene contexto.**

Y para eso ya tenemos la herramienta: es exactamente lo que resolvimos la clase pasada.

---

## 4. El modelo completo

Acá abajo está el bloque transformer **de la clase 7, copiado literalmente**: attention
multi-cabeza con máscara causal, MLP, dos normalizaciones, dos conexiones residuales.
No cambió una coma. Si lo leíste la clase pasada, ya lo entendés.

Lo único nuevo son las tres líneas de `MiniGPT`: la tabla de embeddings de fichas, la
tabla de posiciones, y **la capa final que proyecta al vocabulario**. Esa última capa
es la que convierte «un vector por posición» en «un puntaje para cada ficha posible»,
o sea: en la distribución de probabilidad que el modelo dibuja.

In [ ]:
class AtencionMulticabeza(nn.Module):
    """Cada ficha le pregunta a las anteriores qué tan relevantes son para ella."""

    def __init__(self, dim, cabezas, ctx):
        super().__init__()
        self.cabezas = cabezas
        self.d_cabeza = dim // cabezas          # el ancho se reparte entre las cabezas
        self.proy_q = nn.Linear(dim, dim, bias=False)       # "lo que busco"
        self.proy_k = nn.Linear(dim, dim, bias=False)       # "lo que ofrezco"
        self.proy_v = nn.Linear(dim, dim, bias=False)       # "lo que aporto"
        self.proy_salida = nn.Linear(dim, dim, bias=False)  # mezcla las cabezas
        # un buffer, no un parámetro: la máscara no se aprende, es una regla
        self.register_buffer("mascara", torch.tril(torch.ones(ctx, ctx)))

    def forward(self, x):
        B, T, D = x.shape                       # (lote, fichas, números)
        H, dc = self.cabezas, self.d_cabeza
        q = self.proy_q(x).view(B, T, H, dc).transpose(1, 2)
        k = self.proy_k(x).view(B, T, H, dc).transpose(1, 2)
        v = self.proy_v(x).view(B, T, H, dc).transpose(1, 2)
        puntajes = (q @ k.transpose(-2, -1)) / dc ** 0.5
        # tapar el futuro ANTES del softmax (clase 7, TODO 3)
        puntajes = puntajes.masked_fill(self.mascara[:T, :T] == 0, float("-inf"))
        pesos = torch.softmax(puntajes, dim=-1)              # filas que suman 1
        salida = (pesos @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.proy_salida(salida)


class BloqueTransformer(nn.Module):
    """attention + MLP, cada uno con su normalización y su conexión residual."""

    def __init__(self, dim, cabezas, ctx):
        super().__init__()
        self.norma1 = nn.LayerNorm(dim)
        self.atencion = AtencionMulticabeza(dim, cabezas, ctx)
        self.norma2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),   # se agranda: más rampas para dibujar
            nn.ReLU(),                 # LA rampa de la clase 2
            nn.Linear(4 * dim, dim),   # se vuelve al tamaño de siempre
        )

    def forward(self, x):
        x = x + self.atencion(self.norma1(x))    # residual 1
        return x + self.mlp(self.norma2(x))      # residual 2


print("El bloque de la clase 7, sin tocar. Listo para apilar.")

In [ ]:
class MiniGPT(nn.Module):
    """El bloque de la clase 7, apilado, con una capa final al vocabulario."""

    def __init__(self, vocab, dim, cabezas, capas, ctx):
        super().__init__()
        self.ctx = ctx
        self.emb_ficha = nn.Embedding(vocab, dim)     # una fila por ficha (clase 6)
        self.emb_posicion = nn.Embedding(ctx, dim)    # una fila por posición (clase 7)
        self.bloques = nn.ModuleList(
            [BloqueTransformer(dim, cabezas, ctx) for _ in range(capas)])
        self.norma_final = nn.LayerNorm(dim)
        self.cabeza = nn.Linear(dim, vocab, bias=False)   # -> un puntaje por ficha
        self.apply(self._iniciar)

    @staticmethod
    def _iniciar(modulo):
        if isinstance(modulo, nn.Linear):
            nn.init.normal_(modulo.weight, mean=0.0, std=0.02)
            if modulo.bias is not None:
                nn.init.zeros_(modulo.bias)
        elif isinstance(modulo, nn.Embedding):
            nn.init.normal_(modulo.weight, mean=0.0, std=0.02)

    def forward(self, idx, objetivos=None):
        T = idx.shape[1]
        x = self.emb_ficha(idx) + self.emb_posicion(torch.arange(T, device=idx.device))
        for bloque in self.bloques:
            x = bloque(x)
        logits = self.cabeza(self.norma_final(x))     # (lote, fichas, vocabulario)
        if objetivos is None:
            return logits, None
        # LA LOSS: cross-entropy = "qué probabilidad le diste a la respuesta correcta"
        perdida = F.cross_entropy(
            logits.reshape(-1, logits.shape[-1]), objetivos.reshape(-1))
        return logits, perdida

    @torch.no_grad()
    def generar(self, idx, cuantas, temperatura=1.0, gen=None):
        """Genera fichas de a una. temperatura 0 = siempre la más probable."""
        for _ in range(cuantas):
            recorte = idx[:, -self.ctx:]              # SÓLO lo que entra en la ventana
            logits, _ = self(recorte)
            logits = logits[:, -1, :]                 # nos importa la última posición
            if temperatura <= 0:
                siguiente = logits.argmax(dim=-1, keepdim=True)      # sin sorteo
            else:
                probs = torch.softmax(logits / temperatura, dim=-1)  # con temperatura
                # EL SORTEO. Se hace en CPU, con un generador de CPU: así anda igual
                # con GPU (un generador de CPU sobre un tensor de GPU da error).
                siguiente = torch.multinomial(probs.cpu(), 1, generator=gen)
                siguiente = siguiente.to(idx.device)
            idx = torch.cat([idx, siguiente], dim=1)
        return idx


print("MiniGPT definido.")

### La configuración: el notebook mira si hay GPU y decide

Con GPU sobra tiempo, así que se entrena más. Sin GPU, la configuración está calibrada
para que **el entrenamiento entre en la clase**. Los dos casos entrenan **el mismo
modelo**: cambia sólo cuántos pasos de descenso da.

In [ ]:
hay_gpu = torch.cuda.is_available()
device = torch.device("cuda" if hay_gpu else "cpu")

# La configuración del curso. NO la cambies sin cambiar entrenar-mini-gpt.py:
# las cuatro visualizaciones de la clase salen de una corrida con ESTOS números.
CTX, DIM, CABEZAS, CAPAS, LOTE, LR = 128, 176, 4, 4, 32, 1e-3
PASOS = 3000 if hay_gpu else 1200
CADA = 100 if hay_gpu else 50            # cada cuántos pasos medimos la loss

utils.fijar_semilla(1983)
modelo = MiniGPT(len(tok), DIM, CABEZAS, CAPAS, CTX).to(device)
n_perillas = utils.contar_parametros(modelo)

print(f"device: {device}" + ("" if hay_gpu else f"  ({torch.get_num_threads()} hilos)"))
print(f"modelo: {n_perillas:,} perillas")
print(f"        {CAPAS} bloques · {CABEZAS} cabezas · dim {DIM} · ventana {CTX} fichas")
print(f"pasos:  {PASOS}")
print()
print("Tiempo MEDIDO en el contenedor donde se construyó esta clase:")
print("  CPU (4 hilos):  seis cronometrajes de los mismos 1200 pasos dieron")
print("                  243 · 267 · 280 · 297 · 360 · 432 s -> entre 4 y 7 minutos.")
print("                  La CPU se comparte con todo lo que tengas abierto, así que")
print("                  el número casi se duplica. Planificá con SIETE minutos.")
print("  GPU T4:         NO lo pudimos medir: este contenedor no tiene GPU, así que")
print("                  no vas a encontrar acá un número inventado. Si lo corrés en")
print("                  Colab con T4, anotá lo que imprima la celda de abajo.")
print()
print(f"El modelo de la clase 1 tenía 2 perillas. Este tiene {n_perillas:,}.")
print("Es la misma máquina de dibujar.")

revisar(abs(n_perillas - 1_541_056) < 10,
        f"son las {n_perillas:,} perillas de siempre: el número coincide con el "
        "de las slides y con el de las visualizaciones.",
        f"te dieron {n_perillas:,} y esperábamos 1.541.056. Revisá CTX, DIM, "
        "CABEZAS y CAPAS: alguno quedó cambiado.")

---

## 5. El entrenamiento

Es el **mismo loop de cinco líneas de la clase 4**. Ni una más:

```
predicción = modelo(x)      # 1. PREDECIR
perdida = ...               # 2. COMPARAR
opt.zero_grad()             # 3. BORRAR lo anotado la vuelta pasada
perdida.backward()          # 4. ANOTAR para dónde habría que moverse
opt.step()                  # 5. MOVER las perillas
```

Lo único que agregamos alrededor es **contabilidad**: medir la loss de validación cada
tanto, y **generar una muestra** para poder mirar cómo va aprendiendo.

> ## 🎭 Esto es teatro. Miralo.
>
> Cada tanto el modelo imprime lo que escribiría. La progresión es el momento más lindo
> del curso: **sopa de letras → sílabas → palabras → sintaxis → cadencia
> parlamentaria**. Arrancá la celda y **seguí leyendo la sección 6 mientras corre**;
> volvé cada tanto a mirar las muestras.

> **Tus muestras y las de las slides.** Las slides muestran la *corrida del curso*
> (`entrenar-mini-gpt.py`, en la máquina donde se construyó la clase). En la misma
> computadora y con las mismas versiones de las librerías, esta celda da **exactamente**
> esas muestras. En otra (otra CPU, otra versión de torch, GPU), los pesos cambian en
> las últimas cifras y el sorteo se va por otro camino: la loss va a ser casi la misma,
> pero las muestras de los pasos intermedios van a ser otras. Por ejemplo, la diputada
> inventada del paso 800 de las slides («Nelida Falcon») puede no aparecer en la tuya:
> la celda te avisa qué nombres inventó **tu** modelo. Anotalo: *misma semilla, otra
> máquina, otro texto* es un tema de la clase 10.

In [ ]:
@torch.no_grad()
def perdida_media(modelo, datos_ids, veces=8):
    """La loss promedio sobre `veces` lotes. Una medición, no un paso."""
    modelo.eval()
    total = 0.0
    for _ in range(veces):
        x, y = sacar_lote(datos_ids, LOTE, CTX, gen)
        _, p = modelo(x.to(device), y.to(device))
        total += float(p)
    modelo.train()
    return total / veces


import re

nombres_reales = {d["nombre"] for d in verdad["diputados"]}
patron_nombre = re.compile(r"Diputad[oa] ([A-Z][a-z]+ [A-Z][a-z]+) \(")


def nombres_inventados(txt):
    """Los nombres de encabezado («Diputada Fulana Tal (») que NO están en el padrón."""
    return sorted({m.group(1) for m in patron_nombre.finditer(txt)} - nombres_reales)


def muestra(paso=0, prompt="Señor presidente, ", cuantas=260, temperatura=0.8):
    # La semilla del sorteo es 1983 + el paso: es lo que usó `entrenar-mini-gpt.py`,
    # el script del que salieron las muestras proyectadas en las slides. En la misma
    # computadora y con las mismas versiones de las librerías, el texto que te sale
    # acá es el mismo que hay en pantalla. En otra máquina, con otra versión de torch
    # o con GPU, va a ser parecido, no idéntico.
    modelo.eval()
    gsem = torch.Generator().manual_seed(1983 + paso)   # el sorteo se hace en CPU
    idx = torch.tensor([tok.codificar(prompt)], device=device)
    salida = modelo.generar(idx, cuantas, temperatura, gsem)
    modelo.train()
    return tok.decodificar(salida[0].tolist())


# Reiniciamos el sorteador de lotes. Parece un detalle y no lo es: en la sección 3
# gastamos un lote de ejemplo, y sin esto tu corrida quedaría un pasito corrida
# respecto de la de `entrenar-mini-gpt.py` — que es de donde salen los números de
# las slides y de las visualizaciones. Con esto entrenás el mismo modelo (idéntico
# en la misma máquina y con las mismas versiones de las librerías).
gen = torch.Generator().manual_seed(1983)

opt = torch.optim.AdamW(modelo.parameters(), lr=LR)
historial = {"paso": [], "train": [], "val": []}
hitos = {0, 50, 150, 400, 800, 1200, PASOS}     # los mismos con y sin GPU (TODO 1)
inventados_por_paso = {}


def mostrar_muestra(paso):
    txt = muestra(paso)
    print(txt)
    inventados_por_paso[paso] = nombres_inventados(txt)
    if inventados_por_paso[paso]:
        print(f"   ↑ ojo: {', '.join(inventados_por_paso[paso])} NO está en el padrón. "
              "Se lo inventó.")


print("=" * 72)
print(f"PASO 0 (todavía no aprendió nada) · loss ≈ {math.log(len(tok)):.2f} "
      f"= log({len(tok)}), que es «no tengo ni idea»")
print("=" * 72)
mostrar_muestra(0)
print()

# Medimos la loss del paso 0 antes de tocar nada: es el «no tengo ni idea».
historial["paso"].append(0)
historial["train"].append(perdida_media(modelo, ids_train))
historial["val"].append(perdida_media(modelo, ids_val))
print(f"loss medida en el paso 0: train {historial['train'][0]:.2f} · "
      f"val {historial['val'][0]:.2f}   (repartir parejo entre las {len(tok)} fichas "
      f"daría {math.log(len(tok)):.2f})")
print()

t0 = time.time()
for paso in range(1, PASOS + 1):
    x, y = sacar_lote(ids_train, LOTE, CTX, gen)          # 0. traer un lote
    _, perdida = modelo(x.to(device), y.to(device))       # 1. PREDECIR + 2. COMPARAR
    opt.zero_grad(set_to_none=True)                       # 3. BORRAR
    perdida.backward()                                    # 4. ANOTAR
    opt.step()                                            # 5. MOVER

    if paso % CADA == 0 or paso == PASOS:
        tr, va = perdida_media(modelo, ids_train), perdida_media(modelo, ids_val)
        historial["paso"].append(paso)
        historial["train"].append(tr)
        historial["val"].append(va)
        falta = (time.time() - t0) / paso * (PASOS - paso)
        print(f"paso {paso:5d}/{PASOS} · train {tr:.3f} · val {va:.3f} · "
              f"faltan ~{falta/60:.1f} min", flush=True)

    if paso in hitos:
        print()
        print("=" * 72)
        print(f"PASO {paso}")
        print("=" * 72)
        mostrar_muestra(paso)
        print()

segundos = time.time() - t0
print()
print(f"⏱ ENTRENAMIENTO TERMINADO: {utils.formatear_duracion(segundos)} "
      f"en {device} · {PASOS} pasos · {segundos/PASOS*1000:.0f} ms por paso")

# El chequeo del momento «Nelida Falcon»: ¿tu corrida también inventó a alguien?
todos = sorted({n for v in inventados_por_paso.values() for n in v})
print()
if todos:
    print(f"Nombres inventados en TUS muestras: {', '.join(todos)}. "
          "Ninguno está en el padrón.")
else:
    print("En tus muestras no apareció ningún diputado inventado.")
print("(En la corrida del curso, la muestra del paso 800 trajo «Nelida Falcon», que no")
print(" existe. Si tu máquina es otra, tus muestras también: por eso la sección 9 mide")
print(" la invención sobre 60 sorteos, y no depende de una muestra suelta.)")

---

## 6. Mientras entrena: qué es exactamente la loss

La loss de hoy se llama **cross-entropy** (*entropía cruzada*). Suena a física
estadística y es una cuenta de dos líneas. En castellano:

> **Cross-entropy = qué probabilidad le diste a la respuesta que efectivamente vino.**

Se toma esa probabilidad, se le aplica logaritmo y se le da vuelta el signo. Nada más.

- Le diste 90% a la respuesta correcta → pérdida `-log(0.90) = 0.11`. Casi nada.
- Le diste 10% → pérdida `-log(0.10) = 2.30`. Feo.
- Le diste 1% → pérdida `-log(0.01) = 4.61`. Muy feo.

Fijate en la forma de eso: **castiga muchísimo más estar confiadamente equivocado que
estar dudando**. Si repartís tu probabilidad en partes iguales entre todas las fichas
(o sea: «no tengo idea»), la pérdida es `log(71) = 4.26`. Ese es el número de partida,
y es el que ves en el paso 0. No es un piso: un modelo confiadamente equivocado puede
dar una pérdida **más alta** que no tener idea (lo ves en la tabla de abajo).

Es exactamente el mismo rol que el MSE de la clase 3: un número que dice qué tan mal la
estás dibujando, y que tiene pendiente. Cambia la fórmula porque cambió lo que se
dibuja (antes una recta, ahora una distribución), no cambia la idea.

In [ ]:
# La cuenta, a mano, con tres fichas y nada más.
import math

# Cada caso es (la distribución que produjo el modelo, cuál era la correcta).
casos = {
    "muy seguro y ACIERTA":       ([0.90, 0.07, 0.03], 0),
    "duda, pero acierta":         ([0.40, 0.35, 0.25], 0),
    "no tiene idea (reparte)":    ([0.33, 0.33, 0.34], 0),
    "muy seguro y SE EQUIVOCA":   ([0.90, 0.07, 0.03], 2),
}

print(f"{'situación':<28} {'p(correcta)':>12} {'pérdida':>10}")
print("-" * 52)
perdidas = {}
for nombre, (probs, correcta) in casos.items():
    p = probs[correcta]
    perdidas[nombre] = -math.log(p)
    print(f"{nombre:<28} {p:>12.2f} {perdidas[nombre]:>10.2f}")

razon = perdidas["muy seguro y SE EQUIVOCA"] / perdidas["muy seguro y ACIERTA"]
print()
print("Mirá la primera fila y la cuarta: es LA MISMA distribución (0.90, 0.07, 0.03).")
print(f"Lo único que cambió es cuál era la respuesta correcta, y la pérdida es")
print(f"{razon:.0f} veces más grande.")
print("Por eso el entrenamiento empuja al modelo a NO estar seguro sin motivo:")
print("estar confiadamente equivocado es lo más caro que le puede pasar.")
print()
print(f"Con estas 3 fichas, repartir parejo («no tengo ni idea») da log(3) = "
      f"{math.log(3):.2f}, y la cuarta fila da {perdidas['muy seguro y SE EQUIVOCA']:.2f}:")
print("estar confiadamente equivocado es PEOR que no tener idea. No hay un piso ahí.")
print(f"Con las {len(tok)} fichas del modelo, repartir parejo da log({len(tok)}) = "
      f"{math.log(len(tok)):.2f}: el número que mediste en el paso 0.")

In [ ]:
# La curva de loss. Es la MISMA figura de la clase 4, con la misma lectura.
g.curvas_de_entrenamiento(train=historial["train"], val=historial["val"],
                          titulo="La loss del mini-GPT: azul lo que vio, "
                                 "naranja lo que no",
                          xlabel=f"mediciones (una cada {CADA} pasos de descenso)")
plt.show()

tijera = historial["val"][-1] - historial["train"][-1]
print(f"loss final · train {historial['train'][-1]:.3f} · "
      f"val {historial['val'][-1]:.3f} · diferencia {tijera:.3f}")
print(f"loss del bigrama en validación: {loss_bigrama:.3f}  <- la línea de base")
print()
print("¿Hay tijera? (clase 4: la val sube mientras la train sigue bajando)")
print("  ", "sí, y hay que mirarla" if tijera > 0.15 else
      "no: las dos curvas bajan juntas. Todavía no está calcando.")

### `TODO 1` — leé las muestras como quien lee un corpus

Subí en el notebook y mirá las muestras de los pasos 0, 50, 150, 400, 800 y el final.
Escribí **qué aprendió entre cada par**. No «mejoró»: *qué* mejoró.

**Pista.** Mirá en este orden, que es el orden en que lo aprende:
1. el **largo de las palabras** y dónde caen los espacios;
2. la **puntuación** (¿aparecen los puntos? ¿las comas están en lugares plausibles?);
3. la **estructura de turnos** (¿aparece «Diputado Fulano (SIGLA — Provincia).»?);
4. la **cadencia**: las fórmulas de cortesía, «Nada más, señor presidente».

Y si en alguna muestra aparece un diputado con nombre y apellido, fijate si la celda te
avisó que no está en el padrón.

**Lo que hay que ver.** La forma se aprende **antes** que el contenido. En el paso 800
ya escribe algo que suena exactamente a debate parlamentario y que no significa nada.
Eso no es un defecto de este modelo chiquito: en los modelos grandes esa misma
disociación sigue existiendo, sólo que mucho mejor disimulada.

**Tu respuesta:** *(escribí acá)*

---

## 7. La temperatura: cuánto se arriesga al elegir

Cada vez que el modelo elige una ficha, hay un **sorteo**. La temperatura es la perilla
que decide qué tan cargado está el sorteo:

- **T → 0**: gana siempre la ficha más probable. Determinista, y repetitivo.
- **T = 1**: se sortea con las probabilidades tal cual salieron del modelo.
- **T > 1**: se aplanan las probabilidades y las opciones raras tienen chance.

Lo importante, y es lo que casi todo el mundo entiende mal: **la temperatura no cambia
lo que el modelo sabe**. El orden de las opciones no se mueve. Sólo cambia cuánta
chance le das a las de abajo. Está en `viz/temperatura.html`: movés el slider y las
barras se aplastan, pero la primera sigue siendo la primera.

In [ ]:
# El sorteo, escrito. Miralo línea por línea: son cuatro.
def generar(prompt, cuantas=220, temperatura=0.8, semilla=1983):
    modelo.eval()
    gsem = torch.Generator().manual_seed(semilla)
    idx = torch.tensor([tok.codificar(prompt)], device=device)
    for _ in range(cuantas):
        recorte = idx[:, -CTX:]                      # 1. sólo lo que entra en la ventana
        logits, _ = modelo(recorte)                  # 2. los puntajes de cada ficha
        v = logits[:, -1, :] / max(temperatura, 1e-6)   # 3. la temperatura, acá
        if temperatura <= 0:
            siguiente = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        else:
            probs = torch.softmax(v, dim=-1).cpu()   # el sorteo, en CPU (anda con GPU)
            siguiente = torch.multinomial(probs, 1, generator=gsem).to(device)
        idx = torch.cat([idx, siguiente], dim=1)     # 4. se pega y se vuelve a empezar
    modelo.train()
    return tok.decodificar(idx[0].tolist())


for T in [0.2, 0.8, 1.5]:
    print("=" * 72)
    print(f"TEMPERATURA {T}")
    print("=" * 72)
    print(generar("Señor presidente, ", 220, T))
    print()

### `TODO 2` — encontrá el techo de la temperatura

Probá con temperaturas entre 0,2 y 2,0 y encontrá **la más alta con la que el texto
sigue siendo legible**. Anotala.

Y después la pregunta interesante: **¿cambia según el prompt?** Probá el mismo barrido
arrancando desde `"Se va a votar en general. "` (una fórmula ritual, que el modelo
sabe de memoria) y desde `"Señor presidente, "` (un arranque abierto).

**Cómo saber si te salió.** Vas a encontrar que con un prompt ritual aguanta
temperaturas más altas: el modelo está tan seguro ahí que aplanar la distribución no
alcanza para descarrilarlo. Esa es la respuesta interesante.

In [ ]:
# TODO 2: cambiá la temperatura y el prompt, y anotá dónde se rompe.
MI_TEMPERATURA = 1.2          # TODO: probá 0.2, 0.6, 1.0, 1.4, 1.8, 2.0
MI_PROMPT = "Señor presidente, "   # TODO: probá también "Se va a votar en general. "

print(generar(MI_PROMPT, 200, MI_TEMPERATURA))

### `TODO 3` — por qué el mismo prompt da respuestas distintas

Generá **cinco veces** con el mismo prompt y `T=0.8`, cambiando sólo la semilla.
Después repetí con `T=0.01`.

**Pista.** La semilla es lo único que cambia el resultado del sorteo. Si no hay sorteo
(T≈0), la semilla no cambia nada.

In [ ]:
def completar_corto(prompt, cuantas=20):
    """Completa sin sorteo (T=0) y devuelve SÓLO lo que agregó el modelo."""
    modelo.eval()
    idx = torch.tensor([tok.codificar(prompt)[-CTX:]], device=device)
    with torch.no_grad():
        salida = modelo.generar(idx, cuantas, temperatura=0.0)
    modelo.train()
    return tok.decodificar(salida[0].tolist())[len(tok.codificar(prompt)[-CTX:]):]


# TODO 3: corré esta celda tal cual, mirá la salida, y después cambiá T a 0.01.
T = 0.8            # TODO: probá también 0.01

salidas = [generar("El bloque del ", 90, T, semilla=s) for s in [1, 2, 3, 4, 5]]
for i, s in enumerate(salidas, 1):
    print(f"tirada {i}: {s}")
    print()

distintas = len(set(salidas))
if T <= 0.05:
    revisar(distintas == 1,
            f"con T={T} las cinco son idénticas: casi sin sorteo, la semilla no "
            "cambia nada.",
            f"con T={T} salieron {distintas} textos distintos. Con T tan baja casi "
            "siempre gana la más probable; si te dio distinto, bajala a 0.001.")
else:
    revisar(distintas > 1,
            f"salieron {distintas} textos distintos de 5 con T={T}: cambió sólo el "
            "sorteo.",
            f"con T={T} salieron las cinco iguales. Subí la temperatura (probá 1.0) "
            "o cambiá el prompt por uno más abierto.")
print()
if distintas == 1:
    print("Las cinco son idénticas: sin sorteo, no hay variabilidad. Reproducible.")
else:
    print("Mismo modelo, mismas perillas, mismo prompt. Lo único que cambió fue el")
    print("sorteo. Y esto es exactamente por qué ChatGPT te contesta distinto dos")
    print("veces seguidas — y por qué, si usaste un LLM en tu metodología y no")
    print("reportaste temperatura y semilla, tu análisis NO es reproducible.")

---

## 8. La ventana de contexto

Mirá otra vez la primera línea de `generar`:

```python
recorte = idx[:, -CTX:]      # sólo lo que entra en la ventana
```

Eso es **toda** la ventana de contexto. Lo que quedó afuera **no se le pasa al modelo**.
No es que le preste poca atención: no está. No hay ningún otro lugar donde el modelo
guarde nada entre una ficha y la siguiente.

Vamos a demostrarlo achicando la ventana a mano. Usamos una dependencia real del corpus:
en Aguasclaras, **el nombre de cada diputado predice su bloque** (el modelo aprendió esa
asociación viéndola muchas veces). Así que le damos el encabezado de un turno cortado a
distintos largos y miramos a partir de qué punto puede contestar.

In [ ]:
# Una dependencia que el modelo SÍ aprendió: nombre -> bloque.
# Tomamos encabezados reales del corpus y les vamos achicando la ventana.
import re

patron = re.compile(r"Diputad[oa] ([A-Z][a-z]+ [A-Z][a-z]+) \(([A-Z]{3}) — ([^)]+)\)\.")
casos_enc = []
vistos = set()
for m in patron.finditer(texto):
    if m.group(1) in vistos or m.start() < 300:
        continue
    vistos.add(m.group(1))
    casos_enc.append({"texto": texto[m.start() - 150:m.start(2)],   # hasta el "("
                      "sigla": m.group(2), "nombre": m.group(1),
                      "apellido": m.group(1).split(" ")[1]})

modelo.eval()


def contesta(prefijo, k, cuantas=3):
    """Lo que el modelo escribe viendo SÓLO las últimas k fichas del prefijo."""
    recorte = prefijo[-k:]
    idx = torch.tensor([tok.codificar(recorte)], device=device)
    with torch.no_grad():
        salida = modelo.generar(idx, cuantas, temperatura=0.0)
    return recorte, tok.decodificar(salida[0].tolist())[len(recorte):]


# Elegimos un caso que FALLE con la ventana chica y ACIERTE con la grande — y lo
# decimos, porque con un ejemplo elegido a dedo se puede demostrar cualquier cosa.
# Por eso abajo va, además, la medición sobre los 30.
# (y que siga acertando con la ventana entera: si no, el ejemplo suelto diría lo
#  contrario que la medición sobre los 30, y quedaría un lío en el pizarrón).
c = next((x for x in casos_enc[:30]
          if not contesta(x["texto"], 8)[1].startswith(x["sigla"])
          and contesta(x["texto"], 24)[1].startswith(x["sigla"])
          and contesta(x["texto"], 128)[1].startswith(x["sigla"])), None)
if c is None:      # puede pasar con un modelo poco entrenado: lo decimos y seguimos
    c = casos_enc[0]
    print("(Ningún caso de los 30 falla con 8 y acierta con 24 y con 128: mostramos")
    print(" el primero. Mirá igual la medición sobre los 30, más abajo.)")
    print()

print("EL TEXTO (lo que el modelo puede llegar a ver):")
print("..." + c["texto"][-120:])
print()
print(f"La pregunta: ¿de qué bloque es {c['nombre']}?   "
      f"Respuesta correcta: {c['sigla']}")
print(f"El nombre completo empieza {len(c['texto']) - c['texto'].rindex(c['nombre'])} "
      f"fichas antes del final; el apellido solo, "
      f"{len(c['texto']) - c['texto'].rindex(c['apellido'])}.")
print()

for k in [4, 8, 12, 16, 24, 48, 128]:
    recorte, cont = contesta(c["texto"], k, 4)
    marca = "✓" if cont.startswith(c["sigla"]) else "✗"
    print(f"ventana {k:3d} · ve «...{recorte[-22:]}» · contesta «{cont.strip()}» {marca}")
print()

# Y ahora la medición de verdad, sobre 30 encabezados distintos.
print("Lo mismo, medido sobre 30 diputados distintos (no sobre un caso elegido):")
for k in [4, 8, 12, 16, 24, 32, 64, 128]:
    ok = sum(1 for c2 in casos_enc[:30]
             if contesta(c2["texto"], k)[1].startswith(c2["sigla"]))
    print(f"   ventana {k:3d} fichas -> acierta {ok:2d}/30")
modelo.train()

# Dos líneas de base (clase 4): el azar, y contestar SIEMPRE el bloque más frecuente.
from collections import Counter
mas_comun, n_mas_comun = Counter(c2["sigla"] for c2 in casos_enc[:30]).most_common(1)[0]
print()
print("Para comparar, dos líneas de base que no miran nada:")
print("   tirar a la suerte entre los 6 bloques                  -> 5/30")
print(f"   contestar siempre «{mas_comun}», el bloque más frecuente acá -> "
      f"{n_mas_comun}/30")

**Mirá cómo sube.** Con la ventana en 4 fichas el modelo ve «re (» o algo así, y contesta
lo que contestaría sin mirar: queda al nivel de las líneas de base. Con 8 ve medio
apellido y ya acierta bastante más; con el apellido entero adentro (12 a 16 fichas)
acierta casi siempre. **La curva no es un escalón: sube a medida que entra más del
apellido**, porque cada letra que entra es información.

Lo que sí es un corte duro es **el dato**: la letra que quedó afuera de la ventana no
está borrosa ni «atendida con menos ganas» — no está. Por eso la curva sube de a poco: la
respuesta mejora exactamente a medida que la información entra.

> **Y ahora la parte incómoda, que también está medida.**
> `viz/ventana-de-contexto.html` tiene un segundo caso: la misma pregunta, pero con la
> pista **lejos**, entre 40 y 120 fichas atrás (en el encabezado del turno, y la pregunta
> mucho más adelante). Ahí la curva es **plana**: en la corrida del curso el modelo acierta
> 6 de 24 **incluso con la ventana entera y la pista adentro**. Y contestar *siempre* el
> bloque más frecuente daría 8 de 24. O sea: con la pista lejos, el modelo **no es mejor
> que no mirar la pista**. Contesta su bloque más probable «en general», no el de quien
> habla.
>
> **Estar adentro de la ventana no alcanza.** Este modelo tiene un horizonte efectivo
> bastante más corto que su ventana. En los modelos grandes el fenómeno es más sutil,
> pero es de la misma familia y tiene nombre: *lost in the middle* (Liu et al., 2023).
> Rinden mejor cuando el dato está **al principio o al final** del contexto, y peor
> cuando está en el medio. (La diferencia, dicha con precisión: allá el modelo **podría**
> usar el dato lejano y lo usa peor; acá nunca llegó a aprender esa dependencia larga.)
> **Por eso, en un prompt largo, lo importante va al principio o al final, nunca en el
> medio; y con un documento largo, la pregunta va al final.**
>
> (Esa viz se rehízo después de medir: el plan original mostraba otra cosa que resultó
> ser falsa. Está contado en `notas-docente.md` §9 bis, y vale la pena leerlo: es cómo se
> trabaja.)

**Y por qué la ventana no es infinita.** Attention compara cada ficha con cada otra
ficha: el costo crece con el **cuadrado** de la longitud. El doble de texto son cuatro
veces el cómputo. Ésa es la pregunta que quedó anotada en el cuestionario de la clase 7,
y ésta es la respuesta.

### `TODO 4` — rompé la ventana vos

En el padrón hay **cuatro Falcon**, de cuatro bloques distintos: el apellido solo no
alcanza para saber de qué bloque es. El nombre de pila sí (y el encabezado del turno lo
dice explícito).

La celda de abajo arma un turno: el encabezado de Hector Falcon (VVA), un relleno, y al
final un encabezado nuevo con **sólo el apellido**: «Diputado Falcon (». Tu tarea:

1. Corrila tal cual: el encabezado de arriba está **adentro** de la ventana. Mirá qué
   contesta.
2. Alargá `mi_relleno` hasta que el encabezado quede **afuera** de la ventana de 128
   fichas (la celda te dice a cuántas fichas está).
3. Mirá qué contesta ahora, y comparalo con el padrón que imprime la celda.

**Lo que hay que ver.** Con el encabezado afuera, el modelo no tiene con qué saber cuál
de los cuatro Falcon es. Y contesta igual, **con total soltura**: una sigla y una
provincia, sin ninguna marca de duda. (Y fijate si con el encabezado adentro lo usaba:
arriba viste que, con la pista lejos, este modelo casi no la usa.)

In [ ]:
# TODO 4: alargá `mi_relleno` hasta que el encabezado quede FUERA de la ventana.
mi_relleno = "Señor presidente, " * 2     # TODO: alargalo (probá * 6, * 8, ...)
encabezado = "Diputado Hector Falcon (VVA — Cerro Verde).\n"
mi_texto = encabezado + mi_relleno + "\nDiputado Falcon ("

distancia = len(mi_texto) - mi_texto.index("(")     # el "(" del encabezado de arriba
lo_que_ve = mi_texto[-CTX:]
respuesta = completar_corto(mi_texto, 20)

print(f"el texto entero tiene {len(mi_texto)} fichas")
print(f"el encabezado (con el bloque) está a {distancia} fichas del final "
      f"({'FUERA' if distancia > CTX else 'DENTRO'} de la ventana de {CTX})")
print()
print(f"lo ÚNICO que el modelo va a ver:\n«{lo_que_ve}»")
print()
print("y contesta:", repr(respuesta))
print()

falcons = [d for d in verdad["diputados"] if d["nombre"].endswith(" Falcon")]
print(f"El padrón: hay {len(falcons)} Falcon.")
for d in falcons:
    print(f"   {d['nombre']:<15} {d['partido']} — {d['provincia']}")
elegidos = [d["nombre"] for d in falcons if respuesta.startswith(d["partido"])]
print(f"Con «{respuesta[:3]}» el modelo eligió "
      + (f"a {elegidos[0]}" if elegidos else "un bloque que no tiene ningún Falcon")
      + ". Sin una sola marca de duda.")
print()

revisar(distancia > CTX,
        "el encabezado quedó FUERA de la ventana: el modelo no tiene con qué saber "
        "cuál de los Falcon es, y contestó igual, con total soltura. Ya viste el "
        "mecanismo entero.",
        f"el encabezado todavía está a {distancia} fichas, ADENTRO de la ventana de "
        f"{CTX}: alargá `mi_relleno` hasta que la distancia pase de {CTX}.")

---

## 9. La alucinación, con la verdad al lado

Este es el corazón de la clase.

El modelo **siempre** produce una distribución de probabilidad sobre la próxima ficha.
Siempre. Y adentro no hay nada que **detecte** que no sabe: no hay una verificación, hay
una continuación. Entonces, cuando le preguntás por algo que no está en el corpus,
contesta lo **plausible**.

(«¿Y ChatGPT no dice a veces "no sé"?» Sí: en el post-entrenamiento lo premiaron para que
lo diga más seguido. Pero ese «no sé» también es una continuación plausible que aprendió,
no el resultado de chequear nada. Por eso lo puede decir cuando sí sabe, y no decirlo
cuando inventa.)

Y acá está lo que lo hace peligroso para investigar: **lo plausible y lo verdadero
coinciden la mayoría de las veces**. Por eso la excepción es tan difícil de detectar.

Vamos a ponerle al lado el padrón de la verdad. Van a aparecer **dos fenómenos
distintos**, y conviene no mezclarlos:

- **aceptar una premisa falsa**: le damos nosotros una entidad que no existe (una
  diputada, una ley, una provincia) y el modelo no la rechaza: la completa como si
  existiera;
- **fabricar**: el modelo inventa solo un nombre que nadie le dio. Eso lo medimos al
  final de la sección, sobre 60 sorteos.

In [ ]:
modelo.eval()

def completar(prompt, cuantas=90):
    idx = torch.tensor([tok.codificar(prompt)[-CTX:]], device=device)
    with torch.no_grad():
        salida = modelo.generar(idx, cuantas, temperatura=0.0)   # sin sorteo
    return tok.decodificar(salida[0].tolist())[len(tok.codificar(prompt)[-CTX:]):]


def seguridad(prompt, generado, cuantas=18):
    """Probabilidad típica por ficha de lo que escribió. La 'confianza'."""
    logs, ctx = [], prompt
    for ch in generado[:cuantas]:
        idx = torch.tensor([tok.codificar(ctx)[-CTX:]], device=device)
        with torch.no_grad():
            logits, _ = modelo(idx)
        p = torch.softmax(logits[0, -1, :], dim=-1)[tok.a_id[ch]]
        logs.append(math.log(max(float(p), 1e-9)))
        ctx += ch
    return math.exp(sum(logs) / len(logs))


nombres_reales = {d["nombre"] for d in verdad["diputados"]}

def acierta_todo(d):
    escrito = completar(f"{d['tratamiento'].capitalize()} {d['nombre']} (", 26)
    return (escrito.startswith(d["partido"])
            and escrito.split(")")[0].endswith(d["provincia"]))


# Para el caso "existe" elegimos el primer diputado al que el modelo le acierta el
# bloque Y la provincia (el mismo criterio que usó entrenar-mini-gpt.py para la viz),
# y lo decimos: mostrar sólo los éxitos sería hacer trampa. El número honesto
# (a cuántos de los 60 les acierta) sale en la celda siguiente.
existe = next((d for d in verdad["diputados"] if acierta_todo(d)),
              verdad["diputados"][0])

casos = [
    ("EXISTE en el padrón (le acierta bloque y provincia: elegido a propósito)",
     f"{existe['tratamiento'].capitalize()} {existe['nombre']} ("),
    ("NO existe: el nombre lo pusimos nosotros en el prompt",
     "Diputada Marisol Bengoechea ("),
    ("una ley que no existe: el número lo pusimos nosotros",
     "Cuando discutimos la Ley 7841 de "),
    ("una provincia que no existe: la pusimos nosotros",
     'Los intendentes de Puerto Bermejo nos mandaron una nota. Dice, textual: "'),
]

seguridades = {}
for etiqueta, prompt in casos:
    gen_txt = completar(prompt)
    s = seguridad(prompt, gen_txt)
    seguridades[etiqueta] = s
    print("=" * 72)
    print(etiqueta)
    print(f"  prompt : {prompt}")
    print(f"  escribe: {gen_txt[:80]}")
    print(f"  seguridad del modelo: {s*100:.0f}%")
    print()

print("=" * 72)
print("EL PADRÓN DE LA VERDAD")
print("=" * 72)
print(f"{existe['nombre']}: bloque {existe['partido']}, provincia "
      f"{existe['provincia']}   <- está en el padrón")
_gen_existe = completar(casos[0][1], 26).split(")")[0]
if existe["provincia"] not in _gen_existe:
    print(f"   OJO: el modelo escribió «{_gen_existe}». Le acertó el bloque y le")
    print(f"   erró la provincia — y las dos mitades suenan exactamente igual.")
    print(f"   Media verdad dicha con total soltura: eso es lo difícil de auditar.")
print("Marisol Bengoechea:", "ESTÁ" if "Marisol Bengoechea" in nombres_reales
      else "NO está entre los 60 diputados. No existe.")
print("Ley 7841:", "existe" if any(l["numero"] == "Ley 7841" for l in verdad["leyes"])
      else "NO está entre las 14 leyes del padrón. No existe.")
print("Puerto Bermejo:", "existe" if "Puerto Bermejo" in verdad["provincias"]
      else "NO está entre las 10 provincias. No existe.")
print()
print("Y ahora mirá las cuatro seguridades juntas:")
for etiqueta, s in seguridades.items():
    print(f"   {s*100:5.0f}%   {etiqueta[:56]}")
print()
print("Ninguna te avisa. No hay ningún número, ninguna vacilación, ninguna marca")
print("en la salida que separe el caso verdadero de los tres falsos.")
print()
print("Y mirá qué hizo en los tres falsos: NO rechazó ninguna premisa. Una diputada,")
print("una ley y una provincia que no existen: las completó como si existieran.")
_cita = completar(casos[3][1])[:50]
_veces = texto.count(_cita)
if _veces:
    print(f"Y la «cita» ({_cita}...) no la inventó: está TAL CUAL {_veces} veces")
    print("en el corpus. La recitó de memoria (por eso la seguridad es tan alta) y la")
    print("pegó a una provincia que no existe.")
modelo.train()
print()

In [ ]:
# Un número honesto: ¿cuántos de los 60 diputados que SÍ existen tienen bien
# el bloque (y la provincia), según el modelo? Porque «alucina» no quiere decir
# «no sabe nada».
modelo.eval()
aciertos = aciertos_prov = 0
media_verdad = None
for d in verdad["diputados"]:
    prompt = f"{d['tratamiento'].capitalize()} {d['nombre']} ("
    escrito = completar(prompt, 26).split(")")[0]
    ok_bloque = escrito.startswith(d["partido"])
    ok_prov = escrito.endswith(d["provincia"])
    aciertos += ok_bloque
    aciertos_prov += ok_prov
    if ok_bloque and not ok_prov and media_verdad is None:
        media_verdad = (d, escrito)
modelo.train()

n_dip = len(verdad["diputados"])
print(f"El modelo le acierta el bloque a {aciertos} de {n_dip} diputados reales,")
print(f"y la provincia a {aciertos_prov} de {n_dip}.")
if media_verdad:
    d, escrito = media_verdad
    print(f"Por ejemplo, de {d['nombre']} escribió «{escrito}»: bloque bien, provincia")
    print(f"mal (es de {d['provincia']}). Media verdad, dicha con la misma soltura.")
print()
print("O sea: SABE cosas. El problema no es que no sepa nada.")
print("El problema es que, mirando la salida, no hay manera de saber si este caso")
print("es uno de los que sabe o uno de los que se está inventando.")

### Y ahora sí: ¿inventa nombres solo?

En los tres casos falsos de arriba, la entidad falsa la pusimos nosotros en el prompt.
La otra mitad de la alucinación es que el modelo **fabrique** un nombre sin que nadie se
lo dé. No lo vamos a mostrar con un ejemplo elegido a dedo (el «Nelida Falcon» de las
slides depende de la máquina): le hacemos arrancar 60 encabezados de turno, dejamos que
escriba el nombre, y contamos cuántos están en el padrón.

In [ ]:
# La fabricación espontánea, medida: 60 arranques de un encabezado de turno.
# (El mismo cálculo que hace entrenar-mini-gpt.py, con la misma semilla.)
modelo.eval()
arranque = ("Se va a votar en general. Los que esten por la afirmativa, "
            "sirvanse indicarlo.\nDiputad")
N = 60
idx = torch.tensor([tok.codificar(arranque)] * N, device=device)
gsem = torch.Generator().manual_seed(1983)
with torch.no_grad():
    salida = modelo.generar(idx, 40, temperatura=0.8, gen=gsem)
modelo.train()

escritos = []
for fila in salida:
    m = patron_nombre.match("Diputad" + tok.decodificar(fila[idx.shape[1]:].tolist()))
    if m:
        escritos.append(m.group(1))
inventados = [n for n in escritos if n not in nombres_reales]

print(f"{len(escritos)} de {N} arranques terminaron en un encabezado con nombre y apellido.")
print(f"De esos, {len(inventados)} nombran a alguien que NO está en el padrón "
      f"({len(inventados) / max(len(escritos), 1) * 100:.0f}%).")
if inventados:
    print("Los inventados:", ", ".join(sorted(set(inventados))))
print()
print("Nadie le pidió que inventara. Arrancó un encabezado, y un encabezado lleva un")
print("nombre: puso uno con la forma correcta (nombre, apellido, paréntesis). A veces")
print("existe y a veces no, y la salida no te dice cuál es cuál.")

### `TODO 5` — el ejercicio de criterio (el importante de la clase)

Mirá las cuatro salidas de arriba, el número de seguridad de cada una y la proporción
de nombres inventados.

1. **¿Podrías distinguir, sin conocer el padrón, cuál es la invención?** Contestá
   honestamente.
2. Escribí **tres oraciones** sobre qué implica esto para usar un LLM como fuente de
   información en tu investigación.
3. Y una cuarta: ¿qué tipo de tarea de investigación **sí** podés hacer con una
   herramienta que tiene esta propiedad?

**Para la tercera, la pista que vale la clase entera:** las tareas donde el error es
**visible y barato de verificar** (¿este titular menciona una protesta? lo leo y lo
chequeo en dos segundos) son buenas candidatas. Las tareas donde verificar cuesta tanto
como hacer el trabajo (¿esta cita existe? ¿este dato es correcto?) son las peores.

**Tu respuesta:** *(escribí acá)*

---

## 10. Por qué cuenta mal las letras

Acá cobramos la siembra de la clase 6: aquel panel de `viz/tokenizador.html` que
preguntaba cuántas erres tenía la palabra y se iba sin contestar.

**El mecanismo.** Un modelo grande no ve letras: ve **fichas de sub-palabra**. Con el
tokenizador de GPT, «ferrocarril» son tres o cuatro fichas; con el tokenizador chiquito
del curso (el de la clase 6), seis. Para contar cuántas R tiene, el modelo tiene que
saber **de qué letras está hecha cada ficha**, y eso no está a la vista en su
representación: lo tuvo que aprender aparte, de ver palabras deletreadas.

Los modelos grandes lo aprenden bastante bien: si le pedís que deletree «ferrocarril»,
lo deletrea, y con razonamiento paso a paso (o con una herramienta) cuentan letras y
hacen cuentas de forma bastante confiable. Pero la tokenización lo vuelve **más difícil
y más propenso a error** que para vos, que ves las letras. La lección práctica: **no es
imposible; es frágil. Verificá.**

> **Un detalle honesto, y hay que decirlo.** El mini-GPT que acabás de entrenar es de
> **caracteres**, así que él sí ve las letras: no sufre *este* problema. Sufre otro
> (nunca vio a nadie contar nada en el corpus, así que igual contesta cualquier cosa).
> Usamos caracteres porque con 1 MB de corpus es lo que funciona. Los modelos grandes
> usan sub-palabras porque con caracteres las secuencias se vuelven larguísimas y la
> ventana no alcanza: **es un compromiso, y el conteo de letras es una de las cosas que
> se pagan.**

In [ ]:
# El tokenizador BPE de la clase 6, sobre el corpus real de discursos.
carpeta6 = utils.raiz_del_repo() / "clases" / "06-texto-secuencias-embeddings"
crudo = (carpeta6 / "datos-texto.js").read_text(encoding="utf-8")
T6 = json.loads(crudo[crudo.index("{"):crudo.rindex("}") + 1])
fusiones = {tuple(f): i for i, f in enumerate(T6["bpe"]["fusiones"])}
vocab_bpe = {t: i + 1 for i, t in enumerate(T6["bpe"]["vocabulario"])}


def bpe(palabra):
    """El mismo algoritmo de la clase 6: pegar el par más frecuente, y repetir."""
    piezas = list(palabra) + ["</w>"]
    while True:
        mejor, i_mejor = len(fusiones) + 1, -1
        for i in range(len(piezas) - 1):
            r = fusiones.get((piezas[i], piezas[i + 1]))
            if r is not None and r < mejor:
                mejor, i_mejor = r, i
        if i_mejor < 0:
            return piezas
        piezas[i_mejor:i_mejor + 2] = [piezas[i_mejor] + piezas[i_mejor + 1]]


for palabra in ["ferrocarril", "arroz", "carro", "irresponsabilidad", "sol"]:
    piezas = bpe(palabra)
    ids_bpe = [vocab_bpe.get(p, 0) for p in piezas]
    limpias = [p.replace("</w>", "") for p in piezas]
    print(f"{palabra:<22} tiene {palabra.count('r')} erres")
    print(f"{'':<22} el modelo ve: {limpias}")
    print(f"{'':<22} o sea, ve:    {ids_bpe}")
    print()

print("El modelo no ve esas letras: ve la lista de números. Y aunque en «ferrocarril»")
print("una de las fichas sea justo una «r» suelta, las otras TRES erres están ESCONDIDAS")
print("adentro de «fer», de «ar» y de «ri»: para contarlas tiene que saber de qué letras")
print("está hecha cada ficha, algo que no está a la vista y que tuvo que aprender aparte.")
print("Los modelos grandes lo aprenden bastante (deletrean bien), pero es más frágil de")
print("lo que parece: por eso a veces fallan en algo que para vos es trivial.")
print()
print("Lo mismo pasa con las rimas, los anagramas y la aritmética: «1.234» puede quedar")
print("partido en fichas en lugares raros, y para sumar hay que recomponerlo. Más")
print("difícil y más propenso a error; no imposible. Por eso: verificá.")
print()
pregunta = "Pregunta: ¿cuantas letras R tiene la palabra ferrocarril? Respuesta: tiene "
print("¿Y nuestro mini-GPT, que SÍ ve las letras? Le preguntamos:")
print(f"   {pregunta}", repr(completar_corto(pregunta, 30)))
print("Nunca vio a nadie contar nada en el corpus: contesta con una fórmula de sesión.")

---

## 11. El desafío de a dos: rompé el modelo

**Consigna (15 minutos, de a dos).** Escribí el prompt que rompe el modelo **de la
manera más interesante**. No vale «poner letras al azar»: vale encontrar un caso donde
el modelo haga algo llamativo y donde vos puedas **explicar por qué**, en términos de
las piezas que vimos hoy.

Ideas de las que salieron mejores otros años:

- Un prompt que lo mete en un **bucle** (repite la misma frase para siempre). ¿Por qué?
  Porque cayó en un pozo de alta probabilidad: la ficha más probable después de X es Y,
  y la más probable después de Y es X. Con T baja, no sale nunca.
- Un prompt **fuera de género**: escribile una receta de cocina y mirá cómo lo arrastra
  de vuelta al diario de sesiones a los pocos caracteres.
- Un prompt que lo obligue a **contradecirse** dentro del mismo párrafo.
- Un prompt donde **le pidas que no invente** y lo haga igual.

Después escribí, en dos oraciones, **cuál de las piezas de hoy explica lo que viste**.

In [ ]:
# TODO: tu prompt acá. Cambiá la temperatura para ver cómo cambia lo que pasa.
MI_PROMPT_ROMPEDOR = "Por favor no inventes nada. Decime unicamente la verdad: "

print(generar(MI_PROMPT_ROMPEDOR, 260, temperatura=0.3))

**Qué pieza lo explica:** *(escribí acá)*

---

## 12. Guardar el modelo

Guardar los pesos es guardar **el millón y medio de números que son el modelo**. No hay
un archivo de texto adentro, ni una base de datos, ni un índice: hay números. El archivo
pesa unos 6,4 MB en disco y contiene todo lo que el modelo «sabe».

Eso contesta la pregunta que aparece siempre: *«¿dónde está guardada la información?»*.
Está acá, destilada y mezclada, sin índice y sin manera de consultarla ni de corregirla
puntualmente.

Pero ojo con la conclusión apurada («entonces el texto no está adentro»). Mirá la celda
de abajo: en **este** mini-GPT el modelo pesa más que el corpus, y una parte del corpus
**sí** quedó adentro, tal cual.

In [ ]:
import pathlib

destino = pathlib.Path("mini-gpt-aguasclaras.pt")
torch.save({"pesos": modelo.state_dict(),
            "vocabulario": tok.fichas,
            "config": {"CTX": CTX, "DIM": DIM, "CABEZAS": CABEZAS,
                       "CAPAS": CAPAS, "pasos": PASOS}},
           destino)

mb = destino.stat().st_size / 1e6
print(f"guardado en {destino}  ({mb:.1f} MB)")
print(f"{n_perillas:,} números de 4 bytes = {n_perillas*4/1e6:.1f} MB de pesos "
      "(el resto: las máscaras de atención y el vocabulario).")
print()
print(f"El corpus con el que lo entrenamos pesa {len(texto)/1e6:.1f} MB. "
      f"El modelo, {mb:.1f} MB.")
print("Acá el modelo es MÁS GRANDE que el corpus: le sobra lugar para memorizar.")
print()

# ¿Memorizó? Partimos la muestra final en pedazos de 40 caracteres y buscamos
# cada uno, tal cual, en el corpus.
final = muestra(PASOS)[len("Señor presidente, "):]
pedazos = [final[i:i + 40] for i in range(0, len(final) - 39, 40)]
calcados = [pz for pz in pedazos if pz in texto]
print(f"De los {len(pedazos)} pedazos de 40 caracteres de la muestra final, "
      f"{len(calcados)} están TAL CUAL en el corpus.")
if calcados:
    ejemplo = max(calcados, key=texto.count)       # el pedazo más repetido
    print(f"Por ejemplo, «{ejemplo.replace(chr(10), ' ⏎ ')}» aparece "
          f"{texto.count(ejemplo)} veces en el corpus.")
frase = "Y aca esta el punto: esa frase resume el debate de hoy"
print(f"(Y el remate del párrafo de las slides, «{frase}», aparece "
      f"{texto.count(frase)} veces.)")
print()
print("O sea: este mini-GPT memorizó buena parte de las plantillas con las que se")
print("generó Aguasclaras. En un LLM grande la cuenta da al revés: el texto de")
print("entrenamiento (decenas de terabytes) es muchísimo más grande que los pesos, y")
print("no entra entero. Pero igual memorizan tal cual lo que vieron repetido muchas")
print("veces (Carlini et al., en referencias.md). Lo demás queda mezclado y sin")
print("índice, y lo que no quedó es lo que después se completa con lo plausible.")

---

## 13. Qué te llevás

**Las tres cosas de hoy:**

1. **Un LLM dibuja la probabilidad de la próxima ficha, y nada más.** Es la misma
   máquina de la clase 1, con un millón y medio de perillas en vez de dos. Todo lo que
   hace —lo bueno y lo raro— se sigue de ahí.
2. **Nada adentro detecta que no sabe.** La salida es siempre una distribución, y el
   modelo suena igual de seguro cuando acierta que cuando fabrica. La confianza de un
   LLM no es evidencia de nada.
3. **El prompt es tu instrumento de medición.** Si usaste un LLM para producir un dato
   de tu investigación, el prompt exacto, el modelo, la versión, la fecha y la
   temperatura van en el apéndice. No es burocracia: sin eso, nadie —incluida vos
   dentro de seis meses— puede reproducir lo que hiciste.

---

### La tabla de traducción: cada pieza → un comportamiento que ya viviste

| Pieza del mini-GPT que entrenaste | Comportamiento de ChatGPT que explica |
|---|---|
| La salida es **siempre** una distribución de probabilidad | Alucina; no tiene cómo saber cuándo no sabe (si dice «no sé», también es una continuación aprendida); suena igual de seguro cuando acierta que cuando inventa |
| Está **condicionada por el texto previo** | El prompt cambia todo; asignarle un rol («sos un experto en…») le cambia el registro; el prompt es un instrumento |
| La **ventana de atención es finita** | Se olvida del principio de una conversación larga; ignora instrucciones lejanas; conviene repetir lo importante al final |
| El costo de attention es **cuadrático** | Por eso la ventana tiene un límite y por eso un prompt largo cuesta más caro |
| Hay un **sorteo** al elegir la ficha (temperatura) | El mismo prompt da respuestas distintas; T=0 lo vuelve repetitivo; T alta, delirante; sin semilla no hay reproducibilidad |
| Trabaja con **fichas**, no con letras | Le cuesta contar letras, las rimas, los anagramas y la aritmética (los modelos grandes lo compensan en parte, no del todo); anda peor en idiomas poco representados |
| **No hay estado de creencias**, sólo condicionamiento | Se contradice; si arrancó por un camino equivocado sigue por ahí; se lo puede «convencer» de casi cualquier cosa |
| Entrenado para producir **texto plausible** | Produce referencias bibliográficas con formato impecable y datos inventados: el formato es lo más predecible que hay |
| **Instruction tuning + RLHF** encima (esto tu mini-GPT no lo tiene) | Responde consignas en vez de completar; es amable; a veces adula y te da la razón (el RLHF refuerza eso) |
| Aprendió **del corpus** | Reproduce los sesgos del corpus; sabe mucho de lo que abunda en internet y poco de lo que no |

---

**El vocabulario nuevo** (también está en `recursos/glosario.md`):

| término | en castellano llano |
|---|---|
| *token* / ficha | el pedacito en el que se parte el texto: no es una palabra |
| *cross-entropy* | qué probabilidad le diste a la respuesta que efectivamente vino |
| *temperature* | cuánto se arriesga al sortear la próxima ficha |
| *context window* | cuántas fichas puede mirar hacia atrás; afuera de eso, nada existe |
| *hallucination* | fabricar (o aceptar) una entidad falsa con total confianza, porque es lo plausible |
| *autoregressive* | genera de a una ficha, y cada una condiciona a la siguiente |
| *instruction tuning* | la etapa que lo convierte de autocompletador en asistente |
| *RLHF* | ajustarlo con preferencias humanas; también refuerza la adulación |

**En tu propia investigación, desde ya.**
Si usás un LLM para dar un primer pase de codificación sobre tu corpus:
temperatura 0, prompt fijo y escrito en un archivo, una muestra de al menos 200 casos
validada a mano, y el reporte de modelo + versión + fecha + prompt. Con eso tenés algo
publicable. Sin eso, tenés una anécdota.

**Lo que queda prometido.** Entrenar un modelo desde cero cuesta una fortuna y casi
nunca es lo que necesitás. Lo que vas a hacer de verdad es **agarrar uno que ya entrenó
otro y adaptarlo a tu problema**: eso es la clase 9. Y después, la pregunta más difícil
de todas —cómo convencer a un revisor (y a vos misma) de que lo que medís es lo que
decís que medís— es la clase 10.

---

### Para seguir

Las referencias de la clase están en `referencias.md`. La fuente directa de este
notebook es el *Let's build GPT* de Andrej Karpathy y su repositorio `nanoGPT`: está
citado ahí y corresponde decirlo.